In [ ]:
import json
import re
import pathlib

import numpy as np
import pandas as pd

def read_log_file(path: str | pathlib.Path) -> tuple[str, str]:
    path = pathlib.Path(path)
    raw = path.read_bytes()
    try:
        return raw.decode("utf-8"), "utf-8"
    except UnicodeDecodeError:
        return raw.decode("ISO-8859-1"), "ISO-8859-1"

LOG_PATTERN = re.compile(
    r"^\[(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\]\s+"
    r"\[(?P<register_id>[^\]]+)\]\s+"
    r"item=(?P<item>.+?)\s+"
    r"price=(?P<price>\d+(?:\.\d+)?)\s+"
    r"(?P<currency>[A-Z]{3})\s+"
    r"\((?P<status>SUCCESS|REFUND|CANCELLED)\)\s*$"
)

def parse_log(text: str) -> list[dict]:
    """Extract valid transaction records from log text, ignoring noise lines."""
    records = []
    for line in text.splitlines():
        match = LOG_PATTERN.match(line.strip())
        if match:
            record = match.groupdict()
            record["price"] = float(record["price"])
            records.append(record)
    return records

def remove_price_outliers(prices) -> np.ndarray:
    """Replace values with |x - mean| > 3*std by the median of the valid values."""
    arr = np.asarray(prices, dtype=float).ravel()     # 1D float-array
    mu, sigma = arr.mean(), arr.std()
    is_outlier = np.abs(arr - mu) > 3 * sigma         # boolsk maske
    cleaned = arr.copy()
    if is_outlier.any():
        cleaned[is_outlier] = np.median(arr[~is_outlier])
    return cleaned


def compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray:
    """Element-wise refund ratio with epsilon to avoid division by zero."""
    sales_count = np.asarray(sales_count, dtype=float)
    refund_count = np.asarray(refund_count, dtype=float)
    return refund_count / (sales_count + 1e-6)

